In [1]:
import os
import json
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

from utils import logging_setup, load_text, load_report_directory

load_dotenv();

In [2]:
# Model
MODEL_NAME = "google/medgemma-4b-it"

# Paths
ROOT_DIR = Path.cwd().parent
PROMPT_PATH = Path(ROOT_DIR, "prompts", "prompt.txt")
REPORT_DIR = Path(ROOT_DIR, "reports")
PREDICTIONS_DIR = Path(ROOT_DIR, "predictions")

# Logging and Hugging Face Token
LOGGER = logging_setup()
HF_TOKEN = os.getenv("HF_TOKEN")

if HF_TOKEN is None:
    raise ValueError("HF_TOKEN not found in environment")

# Device
if torch.cuda.is_available():
    device = "cuda"
    LOGGER.info("CUDA is available. Using GPU for inference.")
elif torch.backends.mps.is_available():
    device = "mps"
    LOGGER.info("MPS is available. Using GPU for inference.")
else:
    device = "cpu"
    LOGGER.info("Using CPU for inference.")

2026-07-06 11:41:29,550 [INFO] MPS is available. Using GPU for inference.


In [3]:
_MODEL = None
_TOKENIZER = None

def load_model(model_name: str) -> tuple:
    """Load the model and tokenizer from Hugging Face Hub."""

    global _MODEL, _TOKENIZER

    if _MODEL is not None and _TOKENIZER is not None:
        LOGGER.info("Model already loaded.")
        return _MODEL, _TOKENIZER

    dtype = (
        torch.bfloat16
        if device in {"mps", "cuda"}
        else torch.float32
    )

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=dtype,
        device_map="auto",
        token=HF_TOKEN,
    )

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id

    model.eval()

    _MODEL = model
    _TOKENIZER = tokenizer

    LOGGER.info(f"Model {model_name} loaded successfully.")

    return model, tokenizer

In [4]:
def inference_pipeline(prompt: str, model: AutoModelForCausalLM, tokenizer: AutoTokenizer, max_new_tokens: int = 512) -> dict:
    """Run inference on the model with the given prompt and return the JSON response."""

    # Apply chat template to the prompt
    chat = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False,
        add_generation_prompt=True,
    )

    # Tokenize the input prompt
    inputs = tokenizer(
        chat,
        return_tensors="pt",
    )

    # Move inputs to the same device as the model
    model_device = next(model.parameters()).device

    inputs = {
        k: v.to(model_device)
        for k, v in inputs.items()
    }

    LOGGER.info(f"Running inference on device: {model_device}")

    # Generate the model's response
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )

    # Decode the output and extract the JSON response
    input_len = inputs["input_ids"].shape[1]

    response = tokenizer.decode(
        output[0][input_len:],
        skip_special_tokens=True,
    )

    response = response.strip()

    # Extract JSON from the model's response
    start = response.find("{")
    end = response.rfind("}")

    if start == -1 or end == -1:
        raise ValueError(
            f"No JSON found in model response:\n{response}"
        )

    return json.loads(response[start:end + 1])

In [5]:
def save_results(results: list) -> None:
    """Save the results to a CSV file in the predictions directory."""

    # Create a dataframe from the list of results
    df = pd.json_normalize(results)

    timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")

    LOGGER.info(f"Saving results to: {Path(PREDICTIONS_DIR, f'report_extractions_{timestamp}.csv')}")

    # Save the dataframe to a CSV file in the predictions directory
    df.to_csv(Path(PREDICTIONS_DIR, f"report_extractions_{timestamp}.csv"),
        index=False,
    )

    df.head()

In [6]:
def main(system_prompt: str, reports: dict) -> None:
    """Main function to process reports and extract information using the model."""

    results = []

    model, tokenizer = load_model(MODEL_NAME)

    # Iterate over each report and process it
    for report_id, report_text in reports.items():

        LOGGER.info(f"Processing report: {report_id}")

        # Combine the system prompt and report text into a single prompt for the model
        prompt = f"{system_prompt}\n\n{report_text}"

        try:
            # Run inference and extract the JSON response
            extracted = inference_pipeline(prompt, model, tokenizer)
            extracted["report_id"] = report_id

            results.append(extracted)

        except Exception as e:
            LOGGER.error(f"Failed: {report_id}: {e}")

    # Save the results to a CSV file
    save_results(results)

In [7]:
# Load the system prompt and reports
system_prompt = load_text(PROMPT_PATH)
reports = load_report_directory(REPORT_DIR)

# Run the main function
main(system_prompt, reports)

2026-07-06 11:41:38,374 [INFO] HTTP Request: HEAD https://huggingface.co/google/medgemma-4b-it/resolve/main/config.json "HTTP/1.1 200 OK"
2026-07-06 11:41:38,443 [INFO] HTTP Request: HEAD https://huggingface.co/google/medgemma-4b-it/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
2026-07-06 11:41:38,519 [INFO] HTTP Request: GET https://huggingface.co/api/models/google/medgemma-4b-it/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-07-06 11:41:38,577 [INFO] HTTP Request: GET https://huggingface.co/api/models/google/medgemma-4b-it/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-07-06 11:41:39,428 [INFO] HTTP Request: GET https://huggingface.co/api/models/google/medgemma-4b-it "HTTP/1.1 200 OK"
2026-07-06 11:41:39,491 [INFO] HTTP Request: HEAD https://huggingface.co/google/medgemma-4b-it/resolve/main/config.json "HTTP/1.1 200 OK"
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

2026-07-06 11:41:48,423 [INFO] HTTP Request: HEAD https://huggingface.co/google/medgemma-4b-it/resolve/main/generation_config.json "HTTP/1.1 200 OK"
2026-07-06 11:41:48,426 [INFO] Model google/medgemma-4b-it loaded successfully.
2026-07-06 11:41:48,427 [INFO] Processing report: report001
2026-07-06 11:41:48,437 [INFO] Running inference on device: mps:0
2026-07-06 11:41:53,176 [INFO] Processing report: report002
2026-07-06 11:41:53,179 [INFO] Running inference on device: mps:0
2026-07-06 11:41:57,149 [INFO] Saving results to: /Users/lit7907/Library/CloudStorage/OneDrive-NorthwesternUniversity/Documents/Repos/llm-demo/predictions/report_extractions_20260706_114157.csv
